In [1]:
import nest_asyncio
nest_asyncio.apply()

import sys
sys.path.append("../")

from dotenv import load_dotenv

assert load_dotenv("../.env", override=True), "failed to load .env"

In [2]:
!docker exec crystalvision-ollama-1 ollama pull $OLLAMA_CODE_MODEL

pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest 
pulling d7e4b00a7d7a... 100% ▕████████████████▏  15 GB                         
pulling 109037bec39c... 100% ▕████████████████▏  136 B                         
pulling 097a36493f71... 100% ▕████████████████▏ 8.4 KB                         
pulling 2490e7468436... 100% ▕████████████████▏   65 B                         
pulling d109ef1844fa... 100% ▕████████████████▏  488 B                         
verifying sha256 digest 
writing manifest 
success 


In [3]:
import os
from langchain_ollama import OllamaEmbeddings
from langchain_ollama import ChatOllama

llm = ChatOllama(model=os.getenv("OLLAMA_CODE_MODEL"), temperature=0)
embedding = OllamaEmbeddings(model=os.getenv("OLLAMA_EMBED_MODEL"))

In [4]:
from crystalvision.lang.loaders import explain_database

df = explain_database()
df.head(3)

,code,rarity,cost,category_1,category_2,ex_burst,set,name_en,type_en,job_en,text_en,images,limit_break,icons,element_ja,power,element,numElements
0,1-001H,H,6,X,None,False,[Opus I],Auron,Forward,Guardian,"When Auron deals damage to your opponent, you ...",https://fftcg.cdn.sewest.net/images/cards/full...,False,NaN,"(火,)",9000.0,{Fire},1
1,1-002R,R,5,X,None,False,[Opus I],Auron,Forward,Guardian,The Backups you control cannot be broken by yo...,https://fftcg.cdn.sewest.net/images/cards/full...,False,NaN,"(火,)",9000.0,{Fire},1
2,1-003C,C,2,III,None,False,[Opus I],Red Mage,Backup,Standard Unit,《火》《1》《ダル》: Choose 1 Forward. It cannot block ...,https://fftcg.cdn.sewest.net/images/cards/full...,False,multicard,"(火,)",NaN,{Fire},1


In [5]:
from crystalvision.lang.loaders import CARD_CODE
y = "['8-135H', '23-080R', '19-105H', '1-018L', '4-114L', 'PR-003']"
CARD_CODE.findall(y)

['8-135H', '23-080R', '19-105H', '1-018L', '4-114L', 'PR-003']

In [6]:
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.max_colwidth', None)

In [7]:
from langchain_experimental.agents.agent_toolkits import create_pandas_dataframe_agent
from crystalvision.lang.tools import MultiImageEmbedTool

prefix = "You are a pandas agent. You must work with the DataFrame df where each row represents a unique card. "
for col in df.columns:
    if (col_desc := df[col].attrs.get("description", "")):
        prefix += f"'{col}' refers to {col_desc}. "

prefix += "When you refer to a card, so in a a NAME [CODE] format. Please note paying a Crystal ([C]) is different resource from paying CP, the card name Lightning is different from the element Lightning. You can never overwrite or override the dataframe or df. You can never load or read a csv. You must never make up information."

print(prefix)

agent = create_pandas_dataframe_agent(
    llm,
    df,
    verbose=True,
    include_df_in_prompt=None,
    allow_dangerous_code=True,
    prefix=prefix,
    extra_tools=[MultiImageEmbedTool(df=df)]
)

You are a pandas agent. You must work with the DataFrame df where each row represents a unique card. 'code' refers to the UUID of the card. 'ex_burst' refers to if the card has an exburst/ex burst/ex-burst/ex/EX ability. 'name_en' refers to the English name of the card. 'type_en' refers to the English type of the card. 'images' refers to the card image URL. 'element_ja' refers to each element of the card in Japanese as python set. 'power' refers to the numerical value of power of the card. 'element' refers to each element of the card as python set. 'numElements' refers to the number of elements the card consists of. When you refer to a card, so in a a NAME [CODE] format. Please note paying a Crystal ([C]) is different resource from paying CP, the card name Lightning is different from the element Lightning. You can never overwrite or override the dataframe or df. You can never load or read a csv. You must never make up information.


In [8]:
from langchain_experimental.agents.agent_toolkits.pandas.base import _get_single_prompt

prompt = _get_single_prompt(
    df,
    include_df_in_prompt=None,
    prefix=prefix,
)
prompt

PromptTemplate(input_variables=['agent_scratchpad', 'input', 'tool_names', 'tools'], input_types={}, partial_variables={}, template="You are a pandas agent. You must work with the DataFrame df where each row represents a unique card. 'code' refers to the UUID of the card. 'ex_burst' refers to if the card has an exburst/ex burst/ex-burst/ex/EX ability. 'name_en' refers to the English name of the card. 'type_en' refers to the English type of the card. 'images' refers to the card image URL. 'element_ja' refers to each element of the card in Japanese as python set. 'power' refers to the numerical value of power of the card. 'element' refers to each element of the card as python set. 'numElements' refers to the number of elements the card consists of. When you refer to a card, so in a a NAME [CODE] format. Please note paying a Crystal ([C]) is different resource from paying CP, the card name Lightning is different from the element Lightning. You can never overwrite or override the dataframe

In [9]:
print(prompt.template)

You are a pandas agent. You must work with the DataFrame df where each row represents a unique card. 'code' refers to the UUID of the card. 'ex_burst' refers to if the card has an exburst/ex burst/ex-burst/ex/EX ability. 'name_en' refers to the English name of the card. 'type_en' refers to the English type of the card. 'images' refers to the card image URL. 'element_ja' refers to each element of the card in Japanese as python set. 'power' refers to the numerical value of power of the card. 'element' refers to each element of the card as python set. 'numElements' refers to the number of elements the card consists of. When you refer to a card, so in a a NAME [CODE] format. Please note paying a Crystal ([C]) is different resource from paying CP, the card name Lightning is different from the element Lightning. You can never overwrite or override the dataframe or df. You can never load or read a csv. You must never make up information.

{tools}

Use the following format:

Question: the inpu

In [10]:
x = agent.invoke({"input": "Tell me about 1-001H."})
x



> Entering new AgentExecutor chain...
Thought: I need to find the row in the DataFrame where 'code' is '1-001H'. Then, I can extract information about the card and present it.
Action: python_repl_ast
Action Input: ```python
df[df['code'] == '1-001H']
```     code rarity  cost category_1 category_2  ex_burst       set name_en  \
0  1-001H      H     6          X       None     False  [Opus I]   Auron   

   type_en    job_en  \
0  Forward  Guardian   

                                                                                                    text_en  \
0  When Auron deals damage to your opponent, you may play 1 Fire Backup from your hand onto the field dull.   

                                                         images  limit_break  \
0  https://fftcg.cdn.sewest.net/images/cards/full/1-001H_eg.jpg        False   

  icons element_ja   power element  numElements  
0   NaN       (火,)  9000.0  {Fire}            1  I now know the final answer
Final Answer: 1-001H [Auron] is

{'input': 'Tell me about 1-001H.',
 'output': "1-001H [Auron] is a Forward unit from Opus I with 9000 power and costs 6 CP to play. It belongs to the Guardian job. Auron's ability allows you to play a Fire Backup from your hand onto the field dull when he deals damage to your opponent. He has one element: Fire."}

In [11]:
agent.invoke("Show me images of all fire element Auron cards.")



> Entering new AgentExecutor chain...
Thought: I need to filter the DataFrame for Auron cards with the 'Fire' element and then use MultiImageEmbedTool to display their images.
Action: python_repl_ast
Action Input: ```python
df[ (df['name_en'] == 'Auron') & (df['element'].apply(lambda x: 'Fire' in x))]['code'].tolist()
```['1-001H', '1-002R', '20-002H', '22-001R', '7-001H']Now I have the codes of all Fire element Auron cards. I can use MultiImageEmbedTool to display their images.
Action: MultiImageEmbedTool
Action Input: ['1-001H', '1-002R', '20-002H', '22-001R', '7-001H'](AURON 1-001H)[https://fftcg.cdn.sewest.net/images/cards/full/1-001H_eg.jpg]
(AURON 1-002R)[https://fftcg.cdn.sewest.net/images/cards/full/1-002R_eg.jpg]
(AURON 20-002H)[https://fftcg.cdn.sewest.net/images/cards/full/20-002H_eg.jpg]
(AURON 22-001R)[https://fftcg.cdn.sewest.net/images/cards/full/22-001R_eg.jpg]
(AURON 7-001H)[https://fftcg.cdn.sewest.net/images/cards/full/7-001H_eg.jpg]I now know the final answer
Fina

{'input': 'Show me images of all fire element Auron cards.',
 'output': '(AURON 1-001H)[https://fftcg.cdn.sewest.net/images/cards/full/1-001H_eg.jpg]\n(AURON 1-002R)[https://fftcg.cdn.sewest.net/images/cards/full/1-002R_eg.jpg]\n(AURON 20-002H)[https://fftcg.cdn.sewest.net/images/cards/full/20-002H_eg.jpg]\n(AURON 22-001R)[https://fftcg.cdn.sewest.net/images/cards/full/22-001R_eg.jpg]\n(AURON 7-001H)[https://fftcg.cdn.sewest.net/images/cards/full/7-001H_eg.jpg]'}

In [12]:
agent.invoke("Show me images of 5 random Water element cards.")["output"]



> Entering new AgentExecutor chain...
Thought: I need to filter the DataFrame for Water element cards and then randomly select 5. I can use MultiImageEmbedTool to display the images.
Action: python_repl_ast
Action Input: ```python
import random
water_cards = df[df['element'].apply(lambda x: 'Water' in x)]
random_water_cards = random.sample(list(water_cards['code']), 5)
print(random_water_cards)
```['8-125C/1-170C', '24-125H', '15-125R', '2-122R', '16-110C']
Now I have the codes of 5 random Water cards. I can use MultiImageEmbedTool to display them.
Action: MultiImageEmbedTool
Action Input: ['8-125C/1-170C', '24-125H', '15-125R', '2-122R', '16-110C'] (FAIRY 1-170C)[https://fftcg.cdn.sewest.net/images/cards/full/1-170C_eg.jpg]
(LUNAFREYA 15-125R)[https://fftcg.cdn.sewest.net/images/cards/full/15-125R_eg.jpg]
(CLAVAT 16-110C)[https://fftcg.cdn.sewest.net/images/cards/full/16-110C_eg.jpg]
(AGRIAS 2-122R)[https://fftcg.cdn.sewest.net/images/cards/full/2-122R_eg.jpg]
(TONBERRY & CACTUAR 24

'(FAIRY 1-170C)[https://fftcg.cdn.sewest.net/images/cards/full/1-170C_eg.jpg]\r\n(LUNAFREYA 15-125R)[https://fftcg.cdn.sewest.net/images/cards/full/15-125R_eg.jpg]\r\n(CLAVAT 16-110C)[https://fftcg.cdn.sewest.net/images/cards/full/16-110C_eg.jpg]\r\n(AGRIAS 2-122R)[https://fftcg.cdn.sewest.net/images/cards/full/2-122R_eg.jpg]\r\n(TONBERRY & CACTUAR 24-125H)[https://fftcg.cdn.sewest.net/images/cards/full/24-125H_eg.jpg]'